# RNN


## 0. 준비

In [1]:
import random, sys, urllib.request, os, zipfile
import torch, torch.nn as nn

SEED = 20260912
random.seed(SEED); torch.manual_seed(SEED)
ENV = "colab" if "google.colab" in sys.modules else "local"
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"환경 {ENV} · torch {torch.__version__} · device {DEV}")
if DEV.type == "cpu":
    print("CPU 로 돕니다. 이 노트북의 모든 실습은 CPU 기준으로 설계되어 있습니다.")

OTHERS = [d for d in range(10) if d not in (3, 7)]

def make_order_task(n, length=10, gap=None, seed=None):
    """3 과 7 이 정확히 하나씩. 3 이 먼저면 1, 7 이 먼저면 0."""
    rng = random.Random(seed) if seed is not None else random
    xs, ys = [], []
    for _ in range(n):
        filler = [rng.choice(OTHERS) for _ in range(length - 2)]
        if gap is None:
            i, j = sorted(rng.sample(range(length), 2))
        else:
            i = rng.randrange(0, length - gap); j = i + gap
        y = rng.randint(0, 1)
        a, b = (3, 7) if y == 1 else (7, 3)
        s = filler[:]; s.insert(i, a); s.insert(j, b)
        xs.append(s[:length]); ys.append(y)
    return xs, ys

def make_front_task(n, length, seed=None):
    """3 과 7 을 0번·1번 칸에 둔다. 답은 첫 두 칸에서 정해지고,
    나머지 칸은 전부 상관없는 숫자다. 그래서 거리 = length-2 를
    끝까지 들고 가야 답을 낼 수 있다."""
    rng = random.Random(seed) if seed is not None else random
    xs, ys = [], []
    for _ in range(n):
        y = rng.randint(0, 1)
        a, b = (3, 7) if y == 1 else (7, 3)
        xs.append([a, b] + [rng.choice(OTHERS) for _ in range(length - 2)])
        ys.append(y)
    return xs, ys

def to_tensor(xs):
    X = torch.zeros(len(xs), len(xs[0]), 10)
    for b, s in enumerate(xs):
        for t, d in enumerate(s): X[b, t, d] = 1.0
    return X

def train(model, X, Y, Xe, Ye, epochs=30, lr=3e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr); lf = nn.CrossEntropyLoss()
    for _ in range(epochs):
        p = torch.randperm(len(X))
        for i in range(0, len(X), 128):
            j = p[i:i+128]; opt.zero_grad(); lf(model(X[j]), Y[j]).backward(); opt.step()
    with torch.no_grad():
        return (model(Xe).argmax(1) == Ye).float().mean().item()

xtr, ytr = make_order_task(6000, 10, seed=1)
xte, yte = make_order_task(2000, 10, seed=2)
Ytr, Yte = torch.tensor(ytr), torch.tensor(yte)
print("")
print(f"학습 {len(xtr)}줄 · 평가 {len(xte)}줄")
print("예시 :", xtr[0], "→", "네" if ytr[0] else "아니요")

환경 local · torch 2.14.0 · device cpu
CPU 로 돕니다. 이 노트북의 모든 실습은 CPU 기준으로 설계되어 있습니다.

학습 6000줄 · 평가 2000줄
예시 : [2, 3, 1, 7, 5, 1, 9, 9, 9, 8] → 네


---
## L0 · 합으로 풀어보고 깨뜨리기

넘길 것을 **사람이 정해 봄.** 줄 하나를 숫자 하나로 줄여 MLP 에 넣음. 먼저 합.

In [2]:
def summarize(xs, how):
    if how == "sum":   return torch.tensor([[float(sum(s))] for s in xs])
    if how == "mean":  return torch.tensor([[sum(s)/len(s)] for s in xs])
    if how == "last":  return torch.tensor([[float(s[-1])] for s in xs])
    if how == "first": return torch.tensor([[float(s[0])] for s in xs])

class MLP(nn.Module):
    def __init__(s, i, width=32, depth=2):          # 기본은 은닉층 2개 · 너비 32
        super().__init__()
        layers = [nn.Linear(i, width), nn.ReLU()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), nn.ReLU()]
        s.n = nn.Sequential(*layers, nn.Linear(width, 2))
    def forward(s, x): return s.n(x)

torch.manual_seed(SEED)
acc = train(MLP(1), summarize(xtr, "sum"), Ytr, summarize(xte, "sum"), Yte)
print(f"sum    {acc:.4f}")

sum    0.4880


### 질문을 바꾸면 — 합이 45 이상인가

같은 데이터 · 같은 합 모델 · 질문만 바꿈

In [3]:
sums = [sum(s) for s in xtr]
print("학습 데이터 합의 중앙값 :", sorted(sums)[len(sums) // 2])

ytr45 = torch.tensor([1 if sum(s) >= 45 else 0 for s in xtr])
yte45 = torch.tensor([1 if sum(s) >= 45 else 0 for s in xte])
print(f"평가 데이터에서 합이 45 이상인 줄의 비율 : {yte45.float().mean().item():.3f}")

torch.manual_seed(SEED)
acc = train(MLP(1), summarize(xtr, "sum"), ytr45, summarize(xte, "sum"), yte45)
print(f"합 모델 · '합이 45 이상인가' → {acc:.4f}")

학습 데이터 합의 중앙값 : 45
평가 데이터에서 합이 45 이상인 줄의 비율 : 0.522
합 모델 · '합이 45 이상인가' → 0.9575


### 나머지 세 가지 — 평균 · 마지막 값 · 첫 값

In [4]:
for how in ("mean", "last", "first"):
    torch.manual_seed(SEED)
    acc = train(MLP(1), summarize(xtr, how), Ytr, summarize(xte, how), Yte)
    print(f"{how:<6} {acc:.4f}")

mean   0.4990
last   0.6040
first  0.6085


### 마지막 칸 세어보기

평가 데이터에서 마지막 칸이 3 이나 7 인 줄을 세고, 마지막 칸이 3 도 7 도 아닌 줄을 정답별로 하나씩 보여 줌

In [5]:
n_hit = sum(1 for s in xte if s[-1] in (3, 7))
print(f"마지막 칸이 3 이나 7 인 줄 : {n_hit} / {len(xte)} ({n_hit / len(xte):.3f})")

print("")
print("마지막 칸이 3 도 7 도 아닌 줄 — 정답별로 하나씩")
for want in (1, 0):
    k = next(i for i, s in enumerate(xte) if s[-1] not in (3, 7) and yte[i] == want)
    print(" ", " ".join(map(str, xte[k])), "→", "네" if yte[k] else "아니요")

마지막 칸이 3 이나 7 인 줄 : 422 / 2000 (0.211)

마지막 칸이 3 도 7 도 아닌 줄 — 정답별로 하나씩
  8 8 6 9 5 3 0 7 0 6 → 네
  2 9 8 6 6 6 7 3 9 2 → 아니요


### 모델을 키우면 *(미리 돌린 결과)*

마지막 값 하나만 넘긴 채 층 · 너비 · 학습 횟수를 늘림

In [6]:
X_last, Xe_last = summarize(xtr, "last"), summarize(xte, "last")
print(" 층  너비  epoch   정확도")
for depth, width, ep in ((2, 32, 30), (4, 64, 30), (8, 128, 30), (4, 64, 150), (8, 256, 300)):
    torch.manual_seed(SEED)
    acc = train(MLP(1, width, depth), X_last, Ytr, Xe_last, Yte, epochs=ep)
    print(f" {depth:>2}  {width:>4}  {ep:>5}   {acc:.4f}")

 층  너비  epoch   정확도
  2    32     30   0.6040
  4    64     30   0.5895
  8   128     30   0.6175
  4    64    150   0.6245
  8   256    300   0.4880


### 통째로 펼쳐서 한 번에 넣으면

길이 L 인 줄을 L × 10 칸짜리 한 줄로 펼쳐 MLP 에 넣음 · 길이를 바꿔 봄

In [7]:
def flat(xs, L):
    X = torch.zeros(len(xs), L * 10)
    for b, s in enumerate(xs):
        for t, d in enumerate(s): X[b, t * 10 + d] = 1.0
    return X

def run_flat(L, n_train=6000):
    xa, ya = make_order_task(n_train, L, seed=1)
    xb, yb = make_order_task(2000, L, seed=2)
    torch.manual_seed(SEED)
    model = MLP(L * 10)
    acc = train(model, flat(xa, L), torch.tensor(ya), flat(xb, L), torch.tensor(yb))
    return acc, sum(p.numel() for p in model.parameters())

for L in (10, 20, 50, 100):
    acc, n = run_flat(L)
    print(f"길이 {L:>3} → {acc:.4f} · 파라미터 {n:,}개")

길이  10 → 1.0000 · 파라미터 4,354개
길이  20 → 0.9955 · 파라미터 7,554개
길이  50 → 0.9590 · 파라미터 17,154개
길이 100 → 0.8840 · 파라미터 33,154개


### 데이터 양을 줄이면

길이는 50 으로 고정하고 학습 데이터 양만 줄임

In [8]:
for n_train in (6000, 2000, 500):
    acc, _ = run_flat(50, n_train)
    print(f"길이 50 · 학습 데이터 {n_train:>5,}건 → {acc:.4f}")

길이 50 · 학습 데이터 6,000건 → 0.9590
길이 50 · 학습 데이터 2,000건 → 0.8935
길이 50 · 학습 데이터   500건 → 0.6695


### 봤는가만 넘기면

끝까지 읽은 뒤 '3 을 봤는가 · 7 을 봤는가' 두 가지만 넘김

In [9]:
def seen_two(xs):          # 3 을 봤는가 · 7 을 봤는가
    return torch.tensor([[float(3 in s), float(7 in s)] for s in xs])

torch.manual_seed(SEED)
acc = train(MLP(2, depth=1), seen_two(xtr), Ytr, seen_two(xte), Yte, epochs=40)
print(f"3 을 봤는가 · 7 을 봤는가 → {acc:.4f}")

3 을 봤는가 · 7 을 봤는가 → 0.5120


### 먼저 봤는가를 넘기면

상태 셋 — 아직 둘 다 안 봄 · 3 이 먼저 · 7 이 먼저

In [10]:
def first_seen(xs):        # 끝까지 읽은 뒤에는 '3 이 먼저' 또는 '7 이 먼저' 중 하나
    return torch.tensor([[0.0, 1.0, 0.0] if s.index(3) < s.index(7) else [0.0, 0.0, 1.0] for s in xs])

torch.manual_seed(SEED)
acc = train(MLP(3, depth=1), first_seen(xtr), Ytr, first_seen(xte), Yte, epochs=40)
print(f"어느 쪽을 먼저 봤는가 → {acc:.4f}")

어느 쪽을 먼저 봤는가 → 1.0000


---
## L1 · 하나씩 읽는 모델 만들기

순서를 안 버림. 한 칸씩 읽고, 요약을 다음 칸으로 넘김. 먼저 모델을 만들고 파라미터를 셈.

In [11]:
class Net(nn.Module):
    def __init__(s, kind="RNN", hid=32):
        super().__init__()
        s.r = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[kind](10, hid, batch_first=True)
        s.o = nn.Linear(hid, 2)
    def forward(s, x):
        o, _ = s.r(x); return s.o(o[:, -1, :])

net = Net("RNN")
for L in (10, 50, 200):
    xs, _ = make_order_task(4, L, seed=3)
    out = net(to_tensor(xs))
    print(f"길이 {L:>3} → 출력 {tuple(out.shape)} · 파라미터 {sum(p.numel() for p in net.parameters()):,}개")

길이  10 → 출력 (4, 2) · 파라미터 1,474개
길이  50 → 출력 (4, 2) · 파라미터 1,474개
길이 200 → 출력 (4, 2) · 파라미터 1,474개


### 손으로 펼친 판

In [47]:
torch.manual_seed(SEED)
rnn = nn.RNN(10, 32, batch_first=True)
cell = nn.RNNCell(10, 32)
with torch.no_grad():                    # 다음 셀에서 맞춰 보려고 nn.RNN 의 가중치를 cell 에 그대로 복사해 둔다
    cell.weight_ih.copy_(rnn.weight_ih_l0); cell.weight_hh.copy_(rnn.weight_hh_l0)
    cell.bias_ih.copy_(rnn.bias_ih_l0);     cell.bias_hh.copy_(rnn.bias_hh_l0)

X = to_tensor(xtr[:4])                   # 학습 데이터 네 줄
h = torch.zeros(4, 32); hs = []
for t in range(10):
    h = cell(X[:, t, :], h)              # 이번 칸 숫자 + 직전 요약 → 새 요약
    hs.append(h)
print("h.shape :", h.shape)
print(h)

h.shape : torch.Size([4, 32])
tensor([[ 1.0099e-01, -3.3219e-01,  6.0280e-02,  1.9982e-02, -3.1153e-01,
          1.1244e-02, -1.6596e-01,  9.7651e-02,  3.7926e-01,  2.7567e-01,
         -3.3405e-01, -1.0630e-02,  1.5137e-02, -8.8554e-02, -4.3528e-01,
          3.8340e-02,  9.0629e-02, -1.6103e-01, -2.8356e-01,  2.3050e-01,
          1.9009e-01, -2.2485e-02, -1.8781e-01, -2.1100e-01, -1.2524e-01,
         -2.2825e-01,  1.1310e-01, -4.0570e-02,  2.0633e-01, -2.2111e-01,
          4.0832e-02,  1.1885e-01],
        [-2.1101e-02, -2.6737e-01,  1.4331e-01, -8.8323e-02, -1.0789e-01,
          5.1147e-02, -2.2089e-01,  1.0839e-01,  4.1747e-01,  1.5612e-01,
         -2.3345e-02, -5.4859e-02,  9.5000e-02, -2.8380e-01, -2.2639e-01,
          1.6797e-01,  2.5335e-01, -3.2273e-01, -1.7908e-01,  8.0443e-02,
          1.3828e-01, -1.3057e-01, -8.3104e-03, -1.6083e-01, -1.9610e-01,
         -2.5724e-01,  1.2225e-01, -2.8300e-02,  3.2261e-01, -2.3694e-01,
          1.3057e-01,  8.4345e-02],
        [-

### nn.RNN 과 맞춰 보기

In [13]:
manual = torch.stack(hs, 1)
output, h_n = rnn(X)                     # PyTorch 가 감싸 둔 판
print("(output - manual).abs().max() :", (output - manual).abs().max().item())
print("")
print("x      :", tuple(X.shape))
print("output :", tuple(output.shape))
print("h_n    :", tuple(h_n.shape))
print("")
print("torch.allclose(output[:, -1, :], h_n[0]) :", torch.allclose(output[:, -1, :], h_n[0]))   # 정말 똑같은지는 '정말 같은 값일까' 셀에서

(output - manual).abs().max() : 0.0

x      : (4, 10, 10)
output : (4, 10, 32)
h_n    : (1, 4, 32)

torch.allclose(output[:, -1, :], h_n[0]) : True


### 학습시켜 보기

In [14]:
torch.manual_seed(SEED)
acc = train(Net("RNN"), to_tensor(xtr), Ytr, to_tensor(xte), Yte)
print(f"RNN {acc:.4f}")

RNN 1.0000


### 모양 맞추기 — one_hot · batch_first

In [15]:
import torch.nn.functional as F

digits = torch.tensor([xtr[0]])                  # [1, 10] · 숫자 그대로
rnn = nn.RNN(10, 32, batch_first=True)
x_bad = digits.unsqueeze(-1).float()
print("unsqueeze 로 축만 붙이면 :", tuple(x_bad.shape))
try:
    rnn(x_bad)
except RuntimeError as e:
    print("  에러 :", str(e).splitlines()[0])

x = F.one_hot(digits, num_classes=10).float()
output, h_n = rnn(x)
print("one_hot 으로 열 칸에 적으면 :", tuple(x.shape), "→ output", tuple(output.shape), "· h_n", tuple(h_n.shape))

print("")
x8 = torch.randn(8, 10, 10)
for bf in (True, False):
    output, h_n = nn.RNN(10, 32, batch_first=bf)(x8)
    print(f"batch_first={str(bf):<5}  x {tuple(x8.shape)} · output {tuple(output.shape)} · h_n {tuple(h_n.shape)}")

unsqueeze 로 축만 붙이면 : (1, 10, 1)
  에러 : input.size(-1) must be equal to input_size. Expected 10, got 1
one_hot 으로 열 칸에 적으면 : (1, 10, 10) → output (1, 10, 32) · h_n (1, 1, 32)

batch_first=True   x (8, 10, 10) · output (8, 10, 32) · h_n (1, 8, 32)
batch_first=False  x (8, 10, 10) · output (8, 10, 32) · h_n (1, 10, 32)


### 정말 같은 값일까

`output` 의 마지막 칸과 `h_n` 을 두 방법으로 비교함 · equal 은 비트까지 똑같은가 · allclose 는 아주 작은 오차는 봐주는가

In [16]:
torch.manual_seed(SEED)
rnn = nn.RNN(10, 32, batch_first=True)
output, h_n = rnn(to_tensor(xtr[:8]))
print("torch.equal(output[:, -1, :], h_n[0])    :", torch.equal(output[:, -1, :], h_n[0]))
print("torch.allclose(output[:, -1, :], h_n[0]) :", torch.allclose(output[:, -1, :], h_n[0]))

torch.equal(output[:, -1, :], h_n[0])    : True
torch.allclose(output[:, -1, :], h_n[0]) : True


### UCI HAR 열어보기

폰 센서 기록으로 활동 6가지를 맞히는 공개 데이터 · CC BY 4.0 · 처음 실행 때 내려받음

- 압축 안에 `UCI HAR Dataset.zip` 이 또 있어서 한 번 더 풂
- `X_train.txt` 는 561차원 **요약**이라 시퀀스가 아님 → `Inertial Signals` 폴더의 9개 채널을 씀

In [17]:
URL = "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip"
if not os.path.exists("UCI HAR Dataset"):
    urllib.request.urlretrieve(URL, "har.zip")
    zipfile.ZipFile("har.zip").extract("UCI HAR Dataset.zip")
    zipfile.ZipFile("UCI HAR Dataset.zip").extractall(".")

CH = ["body_acc_x", "body_acc_y", "body_acc_z", "body_gyro_x", "body_gyro_y", "body_gyro_z",
      "total_acc_x", "total_acc_y", "total_acc_z"]

def load_har(split):
    base = os.path.join("UCI HAR Dataset", split)
    chans = []
    for c in CH:
        with open(os.path.join(base, "Inertial Signals", f"{c}_{split}.txt")) as f:
            chans.append([[float(v) for v in line.split()] for line in f])
    X = torch.empty(len(chans[0]), len(chans[0][0]), len(CH))
    for ci, rows in enumerate(chans):
        X[:, :, ci] = torch.tensor(rows)
    with open(os.path.join(base, f"y_{split}.txt")) as f:
        y = torch.tensor([int(v) - 1 for v in f.read().split()])   # 라벨 1–6 → 0–5
    return X, y

Xh, Yh = load_har("train")
Xh_te, Yh_te = load_har("test")
print("X.shape   :", Xh.shape)
print("Xte.shape :", Xh_te.shape)

# 뒤 L4 · 과제 셀에서 쓰는 정의 — 여기서 한 번 실행해 둔다
import time, statistics

mu = Xh.mean((0, 1), keepdim=True); sd = Xh.std((0, 1), keepdim=True)
Xn, Xn_te = (Xh - mu) / sd, (Xh_te - mu) / sd

class HarNet(nn.Module):
    def __init__(s, kind, hid):
        super().__init__()
        s.r = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[kind](9, hid, batch_first=True)
        s.o = nn.Linear(hid, 6)
    def forward(s, x):
        o, _ = s.r(x); return s.o(o[:, -1, :])

def run_har(kind, hid, seed=SEED, Xa=None, Xb=None, epochs=12):
    Xa = Xn if Xa is None else Xa
    Xb = Xn_te if Xb is None else Xb
    random.seed(seed); torch.manual_seed(seed)
    m = HarNet(kind, hid)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3); lf = nn.CrossEntropyLoss()
    t = time.time()
    for _ in range(epochs):
        p = torch.randperm(len(Xa))
        for i in range(0, len(Xa), 64):
            j = p[i:i+64]; opt.zero_grad(); lf(m(Xa[j]), Yh[j]).backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 5.0); opt.step()
    with torch.no_grad():
        acc = (m(Xb).argmax(1) == Yh_te).float().mean().item()
    return round(acc, 4), sum(q.numel() for q in m.parameters()), time.time() - t

def report(kind, hid):
    acc, n, sec = run_har(kind, hid)
    print(f"{kind:<5} hidden {hid:<4} 정확도 {acc:.4f} · 파라미터 {n:>6,} · {sec:.1f}s")

X.shape   : torch.Size([7352, 128, 9])
Xte.shape : torch.Size([2947, 128, 9])


### 그대로 텐서로 만들면

In [18]:
seqs = [[3, 1, 7],
        [9, 7, 0, 5, 3, 2],
        [1, 3, 4, 7]]
try:
    torch.tensor(seqs)
except ValueError as e:
    print("그대로 텐서로 만들면 :", e)

그대로 텐서로 만들면 : expected sequence of length 3 at dim 1 (got 6)


### 세 줄 채워 보기

In [19]:
ML = max(len(s) for s in seqs)
X = torch.zeros(len(seqs), ML, 10)       # 짧은 줄 뒤는 전부 0 인 빈 벡터로 남는다
for b, s in enumerate(seqs):
    for t, d in enumerate(s):
        X[b, t, d] = 1.0
lens = torch.tensor([len(s) for s in seqs])
print("X.shape :", X.shape, "· lens :", lens.tolist())

X.shape : torch.Size([3, 6, 10]) · lens : [3, 6, 4]


### 무엇으로 채웠는지

In [20]:
X0 = X.clone()
X0[0, 3:, 0] = 1.0                        # 첫 줄 빈 칸을 '숫자 0' 으로 채우면
print("빈 벡터로 채움   · 첫 줄 칸별 합 :", X[0].sum(1).int().tolist())
print("숫자 0 으로 채움 · 첫 줄 칸별 합 :", X0[0].sum(1).int().tolist())
print("숫자 0 으로 채운 빈 칸 == 둘째 줄의 진짜 0 칸 :", torch.equal(X0[0, 3], X[1, 2]))

빈 벡터로 채움   · 첫 줄 칸별 합 : [1, 1, 1, 0, 0, 0]
숫자 0 으로 채움 · 첫 줄 칸별 합 : [1, 1, 1, 1, 1, 1]
숫자 0 으로 채운 빈 칸 == 둘째 줄의 진짜 0 칸 : True


---
## L1b · 빈 칸을 읽으면

길이가 5~60 으로 제각각인 줄을 열 칸이 전부 0 인 **빈 벡터**로 채워 한 텐서에 담음.

같은 데이터 · 같은 모델로 요약을 꺼내는 방법만 네 가지로 바꿈.

- 마지막 칸(-1) 읽기 · 실제 마지막 칸(`lens - 1`) 읽기
- 전부 평균 · mask 로 빈 칸을 뺀 평균

In [21]:
def make_ragged(n, lo, hi, seed):
    rng = random.Random(seed); xs, ys = [], []
    for _ in range(n):
        L = rng.randint(lo, hi)
        x, y = make_order_task(1, L, seed=rng.randrange(10**6))
        xs.append(x[0]); ys.append(y[0])
    return xs, ys

def pad(xs, T):                          # 짧은 줄 뒤는 전부 0 인 빈 벡터로 남는다
    X = torch.zeros(len(xs), T, 10)
    for b, s in enumerate(xs):
        for t, d in enumerate(s): X[b, t, d] = 1.0
    return X

class ReadNet(nn.Module):
    def __init__(s, how):
        super().__init__()
        s.r = nn.RNN(10, 32, batch_first=True); s.o = nn.Linear(32, 2); s.how = how
    def forward(s, x, lens):
        out, _ = s.r(x); T = x.shape[1]
        if s.how == "마지막 칸(-1)":
            h = out[:, -1, :]
        elif s.how == "실제 마지막 칸":
            h = out[torch.arange(len(x)), lens - 1]
        elif s.how == "전부 평균":
            h = out.mean(1)
        else:                             # mask 평균
            m = (torch.arange(T)[None, :] < lens[:, None]).float().unsqueeze(-1)
            h = (out * m).sum(1) / m.sum(1)
        return s.o(h)

def compare_reads(lo, hi):
    xa, ya = make_ragged(3000, lo, hi, seed=11)
    xb, yb = make_ragged(1000, lo, hi, seed=12)
    T = max(len(s) for s in xa + xb)
    Xa, Xb = pad(xa, T), pad(xb, T)
    La, Lb = torch.tensor([len(s) for s in xa]), torch.tensor([len(s) for s in xb])
    Ya, Yb = torch.tensor(ya), torch.tensor(yb)
    for how in ("마지막 칸(-1)", "실제 마지막 칸", "전부 평균", "mask 평균"):
        random.seed(SEED); torch.manual_seed(SEED)
        model = ReadNet(how)
        opt = torch.optim.Adam(model.parameters(), lr=3e-3); lf = nn.CrossEntropyLoss()
        for _ in range(25):
            p = torch.randperm(len(Xa))
            for i in range(0, len(Xa), 64):
                j = p[i:i+64]; opt.zero_grad(); lf(model(Xa[j], La[j]), Ya[j]).backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step()
        with torch.no_grad():
            acc = (model(Xb, Lb).argmax(1) == Yb).float().mean().item()
        print(f"{how:<10} {acc:.3f}")

print("길이 5~60")
compare_reads(5, 60)

길이 5~60
마지막 칸(-1)  0.514
실제 마지막 칸   1.000
전부 평균      0.483
mask 평균    1.000


### -1 과 lens - 1

In [22]:
torch.manual_seed(SEED)
output, _ = nn.RNN(10, 32, batch_first=True)(X)
idx = torch.arange(len(X))
print("줄마다 output[:, -1] 과 output[idx, lens - 1] 이 같은가 :",
      [torch.equal(a, b) for a, b in zip(output[:, -1, :], output[idx, lens - 1])])

줄마다 output[:, -1] 과 output[idx, lens - 1] 이 같은가 : [False, True, False]


### 길이 4~12 로 좁히면

In [23]:
print("길이 4~12")
compare_reads(4, 12)

길이 4~12
마지막 칸(-1)  1.000
실제 마지막 칸   1.000
전부 평균      1.000
mask 평균    1.000


---
## L2 · 거리를 늘려 무너뜨리기 *(미리 돌린 결과)*

3 과 7 을 맨 앞에 두고 끝까지의 거리를 늘려 가며 같은 예산(20 epoch)으로 학습함.

In [24]:
# 3 과 7 을 맨 앞에 두고 뒤에 상관없는 숫자를 길게 붙인다.
# 답은 첫 두 칸에서 정해지므로, 그걸 끝까지 들고 가야 한다.
for dist in (8, 48, 78, 98, 198, 398):
    L = dist + 2
    xa, ya = make_front_task(4000, L, seed=1)
    xb, yb = make_front_task(1000, L, seed=2)
    torch.manual_seed(SEED)
    acc = train(Net("RNN"), to_tensor(xa), torch.tensor(ya),
                to_tensor(xb), torch.tensor(yb), epochs=20)
    print(f"거리 {dist:>3} → {acc:.3f}")

거리   8 → 1.000
거리  48 → 1.000
거리  78 → 1.000
거리  98 → 0.517
거리 198 → 0.501
거리 398 → 0.483


### 학습을 세 배 더 시키면 *(미리 돌린 결과)*

In [25]:
for dist in (98, 198, 398):
    xa, ya = make_front_task(4000, dist + 2, seed=1)
    xb, yb = make_front_task(1000, dist + 2, seed=2)
    torch.manual_seed(SEED)
    acc = train(Net("RNN"), to_tensor(xa), torch.tensor(ya),
                to_tensor(xb), torch.tensor(yb), epochs=60)
    print(f"60 epoch · 거리 {dist:>3} → {acc:.3f}")

xa, ya = make_front_task(4000, 100, seed=1)
xb, yb = make_front_task(1000, 100, seed=2)
torch.manual_seed(SEED)
acc = train(Net("RNN"), to_tensor(xa), torch.tensor(ya),
            to_tensor(xb), torch.tensor(yb), epochs=20, lr=1e-2)
print(f"20 epoch · 학습률 1e-2 · 거리  98 → {acc:.3f}")

60 epoch · 거리  98 → 1.000
60 epoch · 거리 198 → 1.000
60 epoch · 거리 398 → 1.000
20 epoch · 학습률 1e-2 · 거리  98 → 1.000


### 앞 → 뒤 · 학습 전 모델

맨 앞 두 칸의 3 · 7 만 맞바꾼 두 줄을 같은 모델에 넣고, 칸마다 두 요약이 얼마나 다른지 잼 (두 요약 차이의 크기 ÷ 요약의 크기 · 512쌍 평균)

In [26]:
from decimal import Decimal

def plain(v):                            # 0.0000077 처럼 소수로 적는다
    return format(Decimal(f"{v:.3g}"), "f")

def whisper(model, L=100, n=512, cols=(1, 2, 5, 10, 20, 40, 70, 99)):
    xs, _ = make_front_task(n, L, seed=31)
    xs_b = [[s[1], s[0]] + s[2:] for s in xs]              # 맨 앞 두 칸만 맞바꿈
    with torch.no_grad():
        oa, _ = model.r(to_tensor(xs)); ob, _ = model.r(to_tensor(xs_b))
    return {t: ((oa[:, t] - ob[:, t]).norm(dim=1) / oa[:, t].norm(dim=1)).mean().item() for t in cols}

torch.manual_seed(SEED)
untrained = Net("RNN")
for t, v in whisper(untrained).items():
    print(f"{t:>2}칸  {plain(v)}")
print("")
print("float32 가 구분하는 가장 작은 상대 차이 :", plain(torch.finfo(torch.float32).eps))

 1칸  0.736
 2칸  0.378
 5칸  0.061
10칸  0.00255
20칸  0.00000767
40칸  0.000000099
70칸  0.000000101
99칸  0.0000000998

float32 가 구분하는 가장 작은 상대 차이 : 0.000000119


### 앞 → 뒤 · 20 epoch 모델 *(미리 돌린 결과)*

거리 98 에서 20 epoch 학습시킨 모델로 같은 차이를 잼

In [27]:
def train_front(dist, epochs):
    xa, ya = make_front_task(4000, dist + 2, seed=1)
    xb, yb = make_front_task(1000, dist + 2, seed=2)
    torch.manual_seed(SEED)
    model = Net("RNN")
    acc = train(model, to_tensor(xa), torch.tensor(ya), to_tensor(xb), torch.tensor(yb), epochs=epochs)
    return model, acc

trained = {}
trained[20], acc = train_front(98, 20)
print(f"20 epoch 모델 · 거리 98 정확도 {acc:.3f}")
for t, v in whisper(trained[20]).items():
    print(f"   {t:>2}칸  {plain(v)}")

20 epoch 모델 · 거리 98 정확도 0.517
    1칸  1.73
    2칸  1.42
    5칸  0.964
   10칸  0.224
   20칸  0.0137
   40칸  0.000151
   70칸  0.000000556
   99칸  0.000000143


### 앞 → 뒤 · 60 epoch 모델 *(미리 돌린 결과)*

In [28]:
trained[60], acc = train_front(98, 60)
print(f"60 epoch 모델 · 거리 98 정확도 {acc:.3f}")
for t, v in whisper(trained[60]).items():
    print(f"   {t:>2}칸  {plain(v)}")

60 epoch 모델 · 거리 98 정확도 1.000
    1칸  1.73
    2칸  1.73
    5칸  1.5
   10칸  1.73
   20칸  1.54
   40칸  1.37
   70칸  1.67
   99칸  1.68


### 학습 없이 손으로 심은 가중치

In [48]:
def handmade(L, scale=0.5):
    torch.manual_seed(SEED)
    cell = nn.RNNCell(10, 8)
    with torch.no_grad():
        cell.weight_ih.zero_(); cell.bias_ih.zero_(); cell.bias_hh.zero_()
        cell.weight_hh.copy_(torch.eye(8))                 # 이전 요약을 그대로 넘김
        cell.weight_ih[0, 3] = scale; cell.weight_ih[0, 7] = -scale   # 3 이면 +0.5 · 7 이면 −0.5
    xs, ys = make_front_task(1000, L, seed=2)
    X, Y = to_tensor(xs), torch.tensor(ys)
    h = torch.zeros(1000, 8)
    with torch.no_grad():
        for t in range(L): h = cell(X[:, t, :], h)
    acc = max(((h[:, 0] > 0).long() == Y).float().mean().item(), ((h[:, 0] < 0).long() == Y).float().mean().item())
    return acc, h[:, 0].abs().mean().item()

for L in (10, 100, 400):
    acc, size = handmade(L)
    print(f"거리 {L - 2:>3} → {acc:.3f} · 끝에 남은 크기 {size:.4f}")

거리   8 → 1.000 · 끝에 남은 크기 0.0377
거리  98 → 1.000 · 끝에 남은 크기 0.0362
거리 398 → 1.000 · 끝에 남은 크기 0.0322


---
## L3 · 기울기가 어디까지 오는가

정확도로는 기억 탓인지 학습 탓인지 못 가림. **기울기를 직접 찍음.**

`nn.RNN` 이 아니라 손으로 펼친 `RNNCell` 을 씀 — 감싸 놓으면 안이 안 보임.

In [30]:
STEPS = (0, 10, 20, 40, 70, 99)

def grad_profile(kind, T=100, hid=32, batch=64, forget_bias=None, dtype=torch.float32, steps=STEPS, model=None):
    torch.manual_seed(SEED)
    cell = {"RNN": nn.RNNCell, "LSTM": nn.LSTMCell}[kind](10, hid).to(dtype)
    head = nn.Linear(hid, 2).to(dtype)
    if model is not None:                        # 학습시킨 Net 의 가중치를 손으로 편 판에 옮긴다
        with torch.no_grad():
            cell.weight_ih.copy_(model.r.weight_ih_l0); cell.weight_hh.copy_(model.r.weight_hh_l0)
            cell.bias_ih.copy_(model.r.bias_ih_l0);     cell.bias_hh.copy_(model.r.bias_hh_l0)
            head.weight.copy_(model.o.weight);          head.bias.copy_(model.o.bias)
    if forget_bias is not None:                  # 네 문의 bias 가 한 텐서에 붙어 있다
        with torch.no_grad():                    #   [ input | forget | cell | output ]
            cell.bias_ih[hid:2*hid].fill_(forget_bias)
            cell.bias_hh[hid:2*hid].fill_(0.0)
    xs, ys = make_front_task(batch, T, seed=7)
    X, Y = to_tensor(xs).to(dtype), torch.tensor(ys)
    h = torch.zeros(batch, hid, dtype=dtype); c = torch.zeros(batch, hid, dtype=dtype); hs = []
    for t in range(T):
        if kind == "RNN": h = cell(X[:, t, :], h)
        else:             h, c = cell(X[:, t, :], (h, c))
        h.retain_grad(); hs.append(h)
    nn.CrossEntropyLoss()(head(hs[-1]), Y).backward()
    return [hs[T-1-b].grad.norm().item() for b in steps]

def show(name, values, fmt=".2e"):               # 한글은 두 칸 폭으로 세어 줄을 맞춘다
    width = sum(2 if ord(ch) > 0x1100 else 1 for ch in name)
    print(name + " " * (14 - width) + "".join(f"{v:>10}" if isinstance(v, int) else f"{v:>10{fmt}}" for v in values))

show("거슬러 간 칸", STEPS)
show("RNN", grad_profile("RNN"))

거슬러 간 칸           0        10        20        40        70        99
RNN             5.16e-02  1.06e-04  2.94e-07  1.33e-12  1.50e-20  0.00e+00


### 0 이 진짜 0 일까

`0.00e+00` 으로 찍힌 값이 정말 0 인지, float32 가 못 적는 것인지 · 같은 계산을 float64 로 나란히 놓음

In [31]:
show("거슬러 간 칸", STEPS)
show("RNN float32", grad_profile("RNN"))
show("RNN float64", grad_profile("RNN", dtype=torch.float64))

every = grad_profile("RNN", steps=range(100))
print("")
print("float32 에서 처음 0 으로 찍히는 자리 :", next(b for b in range(100) if every[b] == 0.0), "스텝 거슬러 간 곳")

거슬러 간 칸           0        10        20        40        70        99
RNN float32     5.16e-02  1.06e-04  2.94e-07  1.33e-12  1.50e-20  0.00e+00
RNN float64     5.16e-02  1.06e-04  2.94e-07  1.33e-12  1.50e-20  3.38e-28

float32 에서 처음 0 으로 찍히는 자리 : 77 스텝 거슬러 간 곳


### 학습시킨 모델의 기울기 *(미리 돌린 결과)*

위 '앞 → 뒤 · 20 epoch 모델' · '앞 → 뒤 · 60 epoch 모델' 셀의 두 모델을 그대로 씀

In [32]:
show("거슬러 간 칸", STEPS)
show("학습 전", grad_profile("RNN"))
for ep in (20, 60):
    show(f"{ep} epoch", grad_profile("RNN", model=trained[ep]))

거슬러 간 칸           0        10        20        40        70        99
학습 전         5.16e-02  1.06e-04  2.94e-07  1.33e-12  1.50e-20  0.00e+00
20 epoch        3.85e-02  1.17e-02  1.20e-03  3.16e-05  1.20e-07  6.63e-10
60 epoch        1.31e-04  2.80e-05  3.52e-05  3.19e-05  3.27e-05  1.45e-04


### output.grad 로 재면

In [33]:
torch.manual_seed(SEED)
rnn = nn.RNN(10, 32, batch_first=True); head = nn.Linear(32, 2)
xs, ys = make_front_task(64, 100, seed=7)
output, h_n = rnn(to_tensor(xs))
output.retain_grad()
nn.CrossEntropyLoss()(head(output[:, -1, :]), torch.tensor(ys)).backward()

g = output.grad
show("거슬러 간 칸", STEPS)
show("output.grad", [g[:, 99 - b, :].norm().item() for b in STEPS])
print("기울기가 정확히 0 인 칸 :", sum(1 for t in range(100) if g[:, t, :].abs().max().item() == 0.0), "/ 100")

거슬러 간 칸           0        10        20        40        70        99
output.grad     5.16e-02  0.00e+00  0.00e+00  0.00e+00  0.00e+00  0.00e+00
기울기가 정확히 0 인 칸 : 99 / 100


### 가중치 행렬을 재면

학습 전 · float64 · 위 L3 과 같은 모델 · 같은 데이터

In [34]:
torch.manual_seed(SEED)
cell = nn.RNNCell(10, 32).to(torch.float64); head = nn.Linear(32, 2).to(torch.float64)
xs, ys = make_front_task(64, 100, seed=7)
X, Y = to_tensor(xs).to(torch.float64), torch.tensor(ys)
h = torch.zeros(64, 32, dtype=torch.float64); hs = []
for t in range(100):
    h = cell(X[:, t, :], h); h.retain_grad(); hs.append(h)
nn.CrossEntropyLoss()(head(hs[-1]), Y).backward()
norms = [hs[t].grad.norm().item() for t in range(100)]

with torch.no_grad():
    W = cell.weight_hh
    radius = torch.linalg.eigvals(W).abs().max().item()
    tanh_d = (1 - torch.stack(hs) ** 2).mean().item()
    print(f"한 번 곱할 때 최대로 늘이는 배율 (최대 특이값) : {torch.linalg.svdvals(W).max().item():.2f}")
    print(f"거듭 곱할 때 남는 배율 (고유값 크기의 최댓값)  : {radius:.2f}")
    print(f"tanh 미분 평균                               : {tanh_d:.2f}")
    print(f"두 값을 곱하면                                : {radius * tanh_d:.2f}")
print(f"실제 기울기에서 잰 한 칸당 비율 (0 → 70스텝)   : {(norms[99 - 70] / norms[99]) ** (1 / 70):.2f}")

한 번 곱할 때 최대로 늘이는 배율 (최대 특이값) : 1.10
거듭 곱할 때 남는 배율 (고유값 크기의 최댓값)  : 0.57
tanh 미분 평균                               : 0.96
두 값을 곱하면                                : 0.55
실제 기울기에서 잰 한 칸당 비율 (0 → 70스텝)   : 0.54


### LSTM 이 돌려주는 것

학습 데이터 여덟 줄을 `nn.LSTM` 에 넣고 돌려주는 것들의 모양을 봄

In [35]:
torch.manual_seed(SEED)
lstm = nn.LSTM(10, 32, batch_first=True)
x = to_tensor(xtr[:8])
output, (h_n, c_n) = lstm(x)
print("x      :", x.shape)
print("output :", output.shape)
print("h_n    :", h_n.shape)
print("c_n    :", c_n.shape)
print("h_n 과 c_n 이 같은 값인가 :", torch.equal(h_n, c_n))

x      : torch.Size([8, 10, 10])
output : torch.Size([8, 10, 32])
h_n    : torch.Size([1, 8, 32])
c_n    : torch.Size([1, 8, 32])
h_n 과 c_n 이 같은 값인가 : False


---
## L3b · 문을 달면 살아날까

같은 측정을 `LSTMCell` 로 함. `LSTM 기본` 은 forget 문의 bias 를 0 으로 둔 것임.

In [36]:
show("거슬러 간 칸", STEPS)
show("RNN",        grad_profile("RNN"), ".1e")
show("LSTM 기본",   grad_profile("LSTM", forget_bias=0.0), ".1e")

거슬러 간 칸           0        10        20        40        70        99
RNN              5.2e-02   1.1e-04   2.9e-07   1.3e-12   1.5e-20   0.0e+00
LSTM 기본        5.3e-02   4.6e-05   4.0e-07   2.8e-11   2.1e-17   0.0e+00


### 문이 얼마나 열려 있었나

forget 문 값을 백 칸 동안 매 칸 기록함 (3 과 7 사이가 99칸인 데이터)

In [37]:
import math

def forget_stats(forget_bias, T=100, hid=32, batch=64):
    torch.manual_seed(SEED)
    cell = nn.LSTMCell(10, hid)
    with torch.no_grad():
        cell.bias_ih[hid:2*hid].fill_(forget_bias); cell.bias_hh[hid:2*hid].fill_(0.0)
    xs, _ = make_order_task(batch, T, gap=T - 1, seed=7)
    X = to_tensor(xs)
    h = torch.zeros(batch, hid); c = torch.zeros(batch, hid); fs = []
    with torch.no_grad():
        for t in range(T):
            g = X[:, t, :] @ cell.weight_ih.T + cell.bias_ih + h @ cell.weight_hh.T + cell.bias_hh
            fs.append(torch.sigmoid(g[:, hid:2*hid]).mean().item())   # 이번 칸 forget 문 값의 평균
            h, c = cell(X[:, t, :], (h, c))
    return sum(fs) / len(fs), sum(math.log10(v) for v in fs)

for b in (0.0, 2.0):
    mean_f, log_prod = forget_stats(b)
    print(f"문에 더하는 값 {b:.0f} · 백 칸 평균 {mean_f:.4f} · 백 칸의 값을 다 곱하면 10^{log_prod:.1f}")

문에 더하는 값 0 · 백 칸 평균 0.4997 · 백 칸의 값을 다 곱하면 10^-30.1
문에 더하는 값 2 · 백 칸 평균 0.8779 · 백 칸의 값을 다 곱하면 10^-5.7


### 문을 열어두면

In [38]:
show("거슬러 간 칸", STEPS)
show("RNN",        grad_profile("RNN"), ".1e")
show("LSTM 기본",   grad_profile("LSTM", forget_bias=0.0), ".1e")
show("LSTM 열어둠", grad_profile("LSTM", forget_bias=2.0), ".1e")
show("LSTM 활짝",   grad_profile("LSTM", forget_bias=4.0), ".1e")

opened = grad_profile("LSTM", forget_bias=2.0)
print("")
print(f"열어둔 경우 0스텝 ÷ 99스텝 : {opened[0] / opened[5]:,.0f}배")

거슬러 간 칸           0        10        20        40        70        99
RNN              5.2e-02   1.1e-04   2.9e-07   1.3e-12   1.5e-20   0.0e+00
LSTM 기본        5.3e-02   4.6e-05   4.0e-07   2.8e-11   2.1e-17   0.0e+00
LSTM 열어둠      5.3e-02   1.0e-03   5.2e-04   1.9e-04   3.9e-05   1.6e-05
LSTM 활짝        5.3e-02   7.2e-04   5.9e-04   6.7e-04   9.8e-04   4.5e-03

열어둔 경우 0스텝 ÷ 99스텝 : 3,303배


---
## L4 · 실제 데이터에서 비교 *(미리 돌린 결과)*

위 'UCI HAR 열어보기' 셀을 먼저 실행함. 입력 9 · 출력 6 · 입력 정규화(학습 데이터의 채널별 평균 · 표준편차) · Adam lr 1e-3 · batch 64 · 12 epoch · `clip_grad_norm_` 5.0 · 실행 시간(초)은 환경마다 다름

In [39]:
report("RNN", 64)
report("LSTM", 64)

RNN   hidden 64   정확도 0.8246 · 파라미터  5,190 · 12.3s
LSTM  hidden 64   정확도 0.9057 · 파라미터 19,590 · 34.7s


### GRU 도 같은 조건으로 *(미리 돌린 결과)*

In [40]:
report("GRU", 64)

GRU   hidden 64   정확도 0.9006 · 파라미터 14,790 · 28.9s


### 시드를 세 번 바꾸면 *(미리 돌린 결과)*

In [41]:
for kind in ("RNN", "LSTM", "GRU"):
    accs = [run_har(kind, 64, seed=s)[0] for s in (SEED, 7, 1234)]
    print(f"{kind:<5} {accs} · 평균 {statistics.mean(accs):.4f} · 폭 {max(accs) - min(accs):.4f}")

RNN   [0.8246, 0.8232, 0.8507] · 평균 0.8328 · 폭 0.0275
LSTM  [0.9057, 0.9013, 0.8962] · 평균 0.9011 · 폭 0.0095
GRU   [0.9006, 0.9009, 0.8863] · 평균 0.8959 · 폭 0.0146


### 파라미터 수를 비슷하게 맞추면

학습 없이 파라미터 수만 셈

In [42]:
for kind, hid in (("RNN", 64), ("RNN", 128), ("GRU", 74), ("LSTM", 64)):
    print(f"{kind:<5} hidden {hid:<4} 파라미터 {sum(p.numel() for p in HarNet(kind, hid).parameters()):>6,}")

RNN   hidden 64   파라미터  5,190
RNN   hidden 128  파라미터 18,566
GRU   hidden 74   파라미터 19,320
LSTM  hidden 64   파라미터 19,590


### 맞춘 크기로 다시 돌리면 *(미리 돌린 결과)*

In [43]:
report("RNN", 128)
report("GRU", 74)

RNN   hidden 128  정확도 0.8317 · 파라미터 18,566 · 16.2s
GRU   hidden 74   정확도 0.9084 · 파라미터 19,320 · 30.7s


### 한 줄 안의 시간 순서를 섞으면 *(미리 돌린 결과)*

같은 창 안에서 128칸의 순서만 섞음(학습 · 평가 모두) · 값의 모음은 그대로임 · LSTM hidden 64

In [44]:
def shuffle_time(X, seed):
    g = torch.Generator().manual_seed(seed); out = X.clone()
    for b in range(X.shape[0]):
        out[b] = X[b][torch.randperm(X.shape[1], generator=g)]
    return out

normal = [run_har("LSTM", 64, seed=s)[0] for s in (SEED, 7, 1234)]
Xs, Xs_te = shuffle_time(Xn, 1), shuffle_time(Xn_te, 2)
shuffled = [run_har("LSTM", 64, seed=s, Xa=Xs, Xb=Xs_te)[0] for s in (SEED, 7, 1234)]
print(f"그대로  {normal} · 평균 {statistics.mean(normal):.4f}")
print(f"섞음    {shuffled} · 평균 {statistics.mean(shuffled):.4f}")
print(f"떨어진 폭 {(statistics.mean(normal) - statistics.mean(shuffled)) * 100:.2f}%p")

그대로  [0.9057, 0.9013, 0.8962] · 평균 0.9011
섞음    [0.8636, 0.8704, 0.8616] · 평균 0.8652
떨어진 폭 3.59%p


### 식으로 보면 — LSTM 과 GRU

아래 식은 PyTorch `nn.LSTM`, `nn.GRU` 문서(torch 2.14)의 표기를 그대로 옮긴 것입니다.

**LSTM**

$$
\begin{array}{ll}
i_t = \sigma(W_{ii} x_t + b_{ii} + W_{hi} h_{t-1} + b_{hi}) \\
f_t = \sigma(W_{if} x_t + b_{if} + W_{hf} h_{t-1} + b_{hf}) \\
g_t = \tanh(W_{ig} x_t + b_{ig} + W_{hg} h_{t-1} + b_{hg}) \\
o_t = \sigma(W_{io} x_t + b_{io} + W_{ho} h_{t-1} + b_{ho}) \\
c_t = f_t \odot c_{t-1} + i_t \odot g_t \\
h_t = o_t \odot \tanh(c_t) \\
\end{array}
$$

**GRU**

$$
\begin{array}{ll}
r_t = \sigma(W_{ir} x_t + b_{ir} + W_{hr} h_{(t-1)} + b_{hr}) \\
z_t = \sigma(W_{iz} x_t + b_{iz} + W_{hz} h_{(t-1)} + b_{hz}) \\
n_t = \tanh(W_{in} x_t + b_{in} + r_t \odot (W_{hn} h_{(t-1)}+ b_{hn})) \\
h_t = (1 - z_t) \odot n_t + z_t \odot h_{(t-1)}
\end{array}
$$

**기호**

- $\sigma$ — sigmoid. 값을 0~1 사이로 만들어, 얼마나 열지 정하는 손잡이로 씁니다.
- $\odot$ — 같은 자리끼리 곱하기입니다.

**강의에서 쓴 이름과 맞춰 보기 — LSTM**

- $i_t$ — 적는 문. 이번 것을 메모에 얼마나 적을지
- $f_t$ — 잊는 문. 메모에서 얼마나 남길지
- $g_t$ — 적을 내용
- $o_t$ — 꺼내는 문
- $c_t = f_t \odot c_{t-1} + i_t \odot g_t$ — 강의에서 말한 덧셈 길입니다
- PyTorch 는 네 bias 를 $i, f, g, o$ 순서로 한 텐서에 붙여 둡니다. 그래서 forget 문의 bias 는 `bias_ih[H:2*H]` 입니다 (H 는 hidden 크기).

**강의에서 쓴 이름과 맞춰 보기 — GRU**

- $r_t$ — 예전 요약을 얼마나 참고할지
- $z_t$ — 예전 요약을 얼마나 남길지. 남기지 않은 $1 - z_t$ 만큼을 새것으로 채웁니다. 두 몫을 더하면 1 입니다.
- $n_t$ — 새 후보
- 문서는 $n_t$ 도 gate 라고 부르지만, 0~1 로 여는 손잡이는 $r_t$ 와 $z_t$ 둘입니다.
- 다른 자료에서는 $z_t$ 와 $1 - z_t$ 의 자리가 바뀐 표기도 있습니다. 이 노트북은 PyTorch 표기를 따릅니다.

### 과제

1. L4 의 HAR 셀들을 직접 다시 돌려 강의 화면의 숫자와 비교함 · 정확도는 같은 환경이면 같게, 환경이 다르면 조금 다르게 나올 수 있음 · 실행 시간(초)은 환경마다 다름
2. 아래 '과제 2' 셀 — hidden 크기를 32 · 128 로 바꿔 파라미터 수와 정확도가 어떻게 바뀌는지 표로 정리함
3. 아래 '과제 3' 셀 — 시드를 두 개 더 넣어 흔들림 폭이 달라지는지 봄 · RNN 과 LSTM · GRU 의 차이가 흔들림보다 큰지 판단함
4. 아래 '과제 4' 셀 — 128칸 중 앞 절반만 섞으면 정확도가 어디쯤 올지 먼저 적은 뒤 빈 곳을 채워 돌려 봄
5. 강의의 아홉 단계 목록을 옆에 두고 진행함 · **막힌 단계 번호**를 적어 둠

**안내**

- 라벨 파일 `y_train.txt` · `y_test.txt` 는 1–6 이라 1 을 빼서 0–5 로 씀 (`load_har` 가 이미 함)
- 결과를 적을 때 출처를 함께 적음 — UCI HAR (Human Activity Recognition Using Smartphones), CC BY 4.0
- 과제 셀은 위 'UCI HAR 열어보기' 셀(`run_har` · `report` · `Xn` 정의)을 먼저 실행해야 돔 · 한 셀에 몇 분씩 걸림

### 과제 2 · hidden 크기 바꾸기

`___` 두 곳에 32 와 128 을 넣음 · 파라미터 수와 정확도를 표로 정리함

In [45]:
for kind in ("RNN", "LSTM", "GRU"):
    for hid in (___, ___):          # 32 와 128
        report(kind, hid)

TypeError: hidden_size should be of type int, got: str

### 과제 3 · 시드 늘리기

`___` 두 곳에 원하는 시드를 넣음 · 폭이 강의의 세 번짜리와 얼마나 달라지는지 봄

In [ ]:
SEEDS5 = (SEED, 7, 1234, ___, ___)   # 시드 두 개를 더 넣음
for kind in ("RNN", "LSTM", "GRU"):
    accs = [run_har(kind, 64, seed=s)[0] for s in SEEDS5]
    print(f"{kind:<5} {accs} · 평균 {statistics.mean(accs):.4f} · 폭 {max(accs) - min(accs):.4f}")

### 과제 4 · 앞 절반만 섞기

돌리기 전에 정확도가 '그대로' 와 '전부 섞음' 사이 어디쯤 올지 적어 둠 · `___` 를 채워 앞 64칸만 섞음 · 위 '한 줄 안의 시간 순서를 섞으면' 셀을 먼저 실행함

In [ ]:
def shuffle_front_half(X, seed):
    g = torch.Generator().manual_seed(seed); out = X.clone()
    half = X.shape[1] // 2
    for b in range(X.shape[0]):
        out[b, :half] = ___            # 앞 half 칸만 섞음 · 위 shuffle_time 을 참고
    return out

Xf, Xf_te = shuffle_front_half(Xn, 1), shuffle_front_half(Xn_te, 2)
front = [run_har("LSTM", 64, seed=s, Xa=Xf, Xb=Xf_te)[0] for s in (SEED, 7, 1234)]
print(f"그대로      평균 {statistics.mean(normal):.4f}")
print(f"앞 절반 섞음 {front} · 평균 {statistics.mean(front):.4f}")
print(f"전부 섞음    평균 {statistics.mean(shuffled):.4f}")